In [24]:
#The purpose of this script is to collect labelled data for 3 classifiers:
#1) An LSTM to recognise when a strikefoot is happening
#2) A Random Forest classifier to determine the strikefoot type (heel, midfoot, forefoot)
#3) A classifier to determine whether overstriding is occurring

#This script combines the functionality of strikefoot_type_trainer and data_extraction_1_2_3.
#A key limitation of data_extraction_1_2_3 was that the "u" frames were not saved. This fixes that.
import cv2
import numpy as np
import json
from numpy_encoder import NumpyEncoder
from video_extracter import extract_keypoints

In [35]:
#Update per video
video_dir = '/Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4'
video_name = 'instavid_8'

#Using extract_keypoints to obtain the essential keypoints from the video
coords = extract_keypoints(video_dir)

left_ankle_arr = coords['left_ankle']
right_ankle_arr = coords['right_ankle']
left_hip_arr = coords['left_hip']
right_hip_arr = coords['right_hip']
left_knee_arr = coords['left_knee']
right_knee_arr = coords['right_knee']
left_shoulder_arr = coords['left_shoulder']
right_shoulder_arr = coords['right_shoulder']
total_frames = coords['Frame Count']
n_detected = len(left_ankle_arr)
print(f'Keypoints detected in {n_detected}/{total_frames} frames')

#The purpose of this block is to extract the following 4 LSTM metrics across ALL frames:
#1) Velocity of the ankle x component
#2) Velocity of the ankle y component
#3) Velocity of the shin (x component of the vector between the knee and the ankle)
#4) Tibial angle
left_ankle_x_vel = np.gradient(left_ankle_arr[:, 0])
left_ankle_y_vel = np.gradient(left_ankle_arr[:, 1])
right_ankle_x_vel = np.gradient(right_ankle_arr[:, 0])
right_ankle_y_vel = np.gradient(right_ankle_arr[:, 1])

#Shin vector goes from ankle to knee
left_shin_vec = left_knee_arr - left_ankle_arr
right_shin_vec = right_knee_arr - right_ankle_arr
left_shin_velocity = np.gradient(left_shin_vec[:, 0])
right_shin_velocity = np.gradient(right_shin_vec[:, 0])

#Tibial angle: -> angle between the shin vector and the vertical (pi/2 subtracted since arctan2 is w.r.t the horizontal)
def _tibial(shin_vec_arr):
    return np.pi / 2 - np.arctan2(shin_vec_arr[:, 1], shin_vec_arr[:, 0])

left_tibial_angle = _tibial(left_shin_vec)
right_tibial_angle = _tibial(right_shin_vec)

#Draws a vertical line upward from the ankle — used to visually judge whether the foot
#lands in front of the pelvis (overstriding) or beneath it (neutral)
def draw_ankle_line(frame, actual_frame_idx, which_leg):

    ankle = left_ankle_arr[actual_frame_idx] if which_leg == 'left' else right_ankle_arr[actual_frame_idx]
    ax, ay = int(ankle[0]), int(ankle[1])
    if ax > 0:
        cv2.line(frame, (ax, ay), (ax, 0), (0, 0, 255), 2)
    return frame

#Calculates all 9 metrics at a given strikefoot frame for the given leg:
#LSTM metrics (4): ankle_x_vel, ankle_y_vel, shin_velocity, tibial_angle
#Classifier 1 metrics (5): knee flexion angle, ankle-hip horizontal offset,
#                           trunk angle, pre-strike ankle velocity, ankle approach angle
#All of these are measured at the strikefoot frame
def compute_all_metrics(actual_frame, which_leg):
    if which_leg == 'left':
        ax_vel = left_ankle_x_vel[actual_frame]
        ay_vel = left_ankle_y_vel[actual_frame]
        sh_vel = left_shin_velocity[actual_frame]
        tib = left_tibial_angle[actual_frame]
        ankle_arr = left_ankle_arr
        knee_arr = left_knee_arr
        hip_arr = left_hip_arr
        shldr_arr = left_shoulder_arr
        all_ay = left_ankle_y_vel
    else:
        ax_vel = right_ankle_x_vel[actual_frame]
        ay_vel = right_ankle_y_vel[actual_frame]
        sh_vel = right_shin_velocity[actual_frame]
        tib = right_tibial_angle[actual_frame]
        ankle_arr = right_ankle_arr
        knee_arr = right_knee_arr
        hip_arr = right_hip_arr
        shldr_arr = right_shoulder_arr
        all_ay = right_ankle_y_vel

    #Knee flexion
    #np.arctan2 gives angles relative to the horizontal, so the angles must be subtracted
    v1 = hip_arr[actual_frame] - knee_arr[actual_frame]
    v2 = ankle_arr[actual_frame] - knee_arr[actual_frame]
    knee_flexion = float(np.rad2deg(np.arctan2(v2[1], v2[0]) - np.arctan2(v1[1], v1[0])) % 360)

    #Ankle-hip horizontal offset
    offset = float(np.abs(ankle_arr[actual_frame, 0] - hip_arr[actual_frame, 0]))

    #Trunk angle: -> angle between the shoulder and the hip w.r.t the vertical
    tv = shldr_arr[actual_frame] - hip_arr[actual_frame]
    trunk_angle = float(np.rad2deg(np.pi / 2 - np.arctan2(tv[1], tv[0])))

    #Pre-strikefoot ankle velocity (2 detected frames before the strikefoot)
    pre2 = max(0, actual_frame - 2)
    ankle_vel_pre = float(all_ay[pre2])

    #Pre-strikefoot ankle movement direction angle
    p1 = max(0, actual_frame - 1)
    p3 = max(0, actual_frame - 3)
    dx = float(ankle_arr[p1, 0] - ankle_arr[p3, 0])
    dy = float(ankle_arr[p1, 1] - ankle_arr[p3, 1])
    approach_angle = float(np.arctan2(dy, dx))

    return {
        'ankle_x_vel': float(ax_vel),
        'ankle_y_vel': float(ay_vel),
        'shin_velocity': float(sh_vel),
        'tibial_angle': float(tib),
        'knee_flexion_angle': knee_flexion,
        'ankle_hip_horizontal_offset': offset,
        'trunk_angle': trunk_angle,
        'ankle_velocity_pre_strike': ankle_vel_pre,
        'ankle_approach_angle': approach_angle,
    }


video 1/1 (frame 1/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 44.6ms
video 1/1 (frame 2/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 41.0ms
video 1/1 (frame 3/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 40.5ms
video 1/1 (frame 4/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 41.7ms
video 1/1 (frame 5/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 42.3ms
video 1/1 (frame 6/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 40.3ms
video 1/1 (frame 7/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 40.0ms
video 1/1 (frame 8/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 41.4ms
video 1/1 (frame 9/288) /Users/abhinavarora/Desktop/CadenceCV/Videos/instavid_8.mp4: 640x384 1 person, 44.2ms
video 1/1

In [32]:
#This function will go through the video frame by frame. Each frame is classified as:
#"u" = the EXACT first frame where the foot begins going up (toe-off)
#"d" = the EXACT first frame where the foot contacts the ground (strikefoot)
#"s" = skip (if the frame is neither u nor d)
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively.
#Each consecutive pair of "u" and "d" will be classified as a gait cycle.
def scrub_and_label(video_dir, which_leg):
    side_char = 'L' if which_leg == 'left' else 'R'
    cap = cv2.VideoCapture(video_dir)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    raw_labels = []
    current_u_frame = None

    for frame_idx in range(total):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            break

        frame = draw_ankle_line(frame, frame_idx, which_leg)

        #Helps in identifying which foot is to be tracked
        h, w = frame.shape[:2]
        # Scale font size relative to frame width (1280 used as a reference baseline)
        scale = w / 1280
        fs = scale * 0.55
        th = max(1, int(scale * 2))
        mg = int(scale * 15)

        cv2.putText(frame, f'{which_leg.upper()} | Frame {frame_idx}',
                    (mg, mg + 24), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 0), th, cv2.LINE_AA)
        if current_u_frame is not None:
            cv2.putText(frame, f'u_frame={current_u_frame}',
                        (mg, mg + 50), cv2.FONT_HERSHEY_SIMPLEX, fs, (200, 200, 0), th, cv2.LINE_AA)
        cv2.putText(frame, 'u=up  d=down  s=skip  c=quit',
                    (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 255, 0), th, cv2.LINE_AA)

        cv2.imshow('Labelling', frame)
        key = cv2.waitKey(0) & 0xFF

        #Constructing a single gait cycle — u_frame is stored until the matching d completes it
        if key == ord('u'):
            current_u_frame = frame_idx

        elif key == ord('d'):
            #"d" is always the strikefoot frame — immediately prompt for strike type then overstriding
            strike_frame = frame.copy()
            #Input classification (h = heel, m = midfoot, f = forefoot, s = skip this strike entirely)
            cv2.putText(strike_frame, 'Strike: h=heel  m=midfoot  f=forefoot  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 165, 255), th, cv2.LINE_AA)
            cv2.imshow('Labelling', strike_frame)
            pk = cv2.waitKey(0) & 0xFF

            if pk == ord('s'):
                current_u_frame = None
                continue

            strike_map = {ord('h'): 'heel', ord('m'): 'midfoot', ord('f'): 'forefoot'}
            if pk not in strike_map:
                current_u_frame = None
                continue
            strike_pattern = strike_map[pk]

            #Input overstriding classification — the vertical ankle line helps judge this
            #If the line falls anterior to the pelvis it is overstriding, if within it is neutral
            over_frame = frame.copy()
            cv2.putText(over_frame, 'Overstride: o=yes  n=neutral  s=skip',
                        (mg, h - mg), cv2.FONT_HERSHEY_SIMPLEX, fs, (0, 0, 255), th, cv2.LINE_AA)
            cv2.imshow('Labelling', over_frame)
            ok_key = cv2.waitKey(0) & 0xFF

            overstride_map = {ord('o'): 'overstriding', ord('n'): 'neutral', ord('s'): 'skip'}
            overstriding = overstride_map.get(ok_key, 'skip')

            #Appending the completed gait cycle and resetting u_frame for the next cycle
            raw_labels.append({
                'frame': frame_idx,
                'side': side_char,
                'strike_pattern': strike_pattern,
                'overstriding': overstriding,
                'u_frame': current_u_frame,
            })
            current_u_frame = None

        elif key == ord('c'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return raw_labels

#These metrics must be extracted for ALL frames in each video before scrubbing begins
#Manual labelling done via tracking the left foot first
left_raw  = scrub_and_label(video_dir, 'left')
#This time, the right foot will be focused on to identify its gait cycles
right_raw = scrub_and_label(video_dir, 'right')

In [33]:
#Obtaining metrics for each individual strikefoot frame and combining with the raw labels
#left and right entries are merged into a single list and sorted by frame number
all_entries = []

for raw in left_raw + right_raw:
    which_leg = 'left' if raw['side'] == 'L' else 'right'
    metrics = compute_all_metrics(raw['frame'], which_leg)
    entry = {
        'frame': raw['frame'],
        'side': raw['side'],
        'strike_pattern': raw['strike_pattern'],
        'overstriding': raw['overstriding'],
        'u_frame': raw['u_frame'],
        'video': video_name,
        **metrics,
    }
    all_entries.append(entry)

all_entries.sort(key=lambda x: (x['frame'], x['side']))
print(f'Total labeled strikes: {len(all_entries)}')

Total labeled strikes: 82


In [34]:
#Data loading loop:
#1) Read existing data from strikefoot_data.json
#2) To the same list, add the new entries for this video
#3) Finally, rewrite strikefoot_data.json with the combined data
#If the file doesn't exist or holds old-format data (a dict), start fresh as an empty list
json_path = '/Users/abhinavarora/Desktop/CadenceCV/Json Data/strikefoot_data.json'
with open(json_path, 'r') as f:
    existing = json.load(f)

existing.extend(all_entries)
with open(json_path, 'w') as f:
    json.dump(existing, f, cls=NumpyEncoder, indent=4)
